In [4]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import sqlite3
con = sqlite3.connect("../data/social.db")
tables = {"posts": "../data/social_posts.csv", "brands": "../data/brands.csv",
          "orders": "../data/gate_orders.csv", "customers": "../data/gate_customers.csv"}
for name, path in tables.items():
    pd.read_csv(path).to_sql(name, con, index=False, if_exists="replace")

toys = {
    "toy": pd.DataFrame({"name": ["a","b","c","d","e"], "team": ["x","y","x","y","x"],
                         "score": [5, 12, None, 20, 8], "city": ["UK","US","UK",None,"US"]}),
    "teams": pd.DataFrame({"team": ["x","y","z"], "coach": ["Kim","Lee","Ray"]}),
    "sponsors": pd.DataFrame({"team": ["x","x","y"], "sponsor": ["Nike","Puma","Adidas"]}),
    "staff": pd.DataFrame({"emp_id": [1,2,3,4,5], "name": ["Ana","Ben","Cat","Dev","Eli"],
                           "manager_id": [None, 1, 1, 2, 9]}),
    "exam": pd.DataFrame({"name": ["Amy","Bob","Cal","Dan","Eve","Fay","Gus"],
                          "class": ["A","A","A","A","B","B","B"],
                          "mark": [90, 85, 85, 70, 95, 80, 80]}),
    "sales": pd.DataFrame({"store": ["S1"]*4 + ["S2"]*4, "day": [1,2,3,4]*2,
                           "amount": [100, 150, 120, 200, 80, 80, 90, 60]}),
}
for name, df in toys.items():
    df.to_sql(name, con, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql(sql, con)

# import os
# if os.path.basename(os.getcwd()) != "Lesson_3":
#     os.chdir("Lesson_3")
# os.getcwd()

300

5

420

120

5

3

3

5

7

8

FUNCTION() OVER (PARTITION BY group_col ORDER BY sort_col)

SUM(mark) OVER ()                      -- grand total on every row
100.0 * mark / SUM(mark) OVER ()      -- each row's % of total
mark - AVG(mark) OVER (PARTITION BY class)   -- how far from the class average

SELECT name, mark,
       ROW_NUMBER() OVER (ORDER BY mark DESC) AS rn,
       RANK()       OVER (ORDER BY mark DESC) AS rk,
       DENSE_RANK() OVER (ORDER BY mark DESC) AS dr
FROM exam

WITH ranked AS (
    SELECT name, class, mark,
           ROW_NUMBER() OVER (PARTITION BY class ORDER BY mark DESC) AS rn
    FROM exam
)
SELECT * FROM ranked WHERE rn <= 2

FROM → JOIN → WHERE → GROUP BY → HAVING → **SELECT (windows here)** → ORDER BY → LIMIT

SELECT store, day, amount,
       LAG(amount)  OVER (PARTITION BY store ORDER BY day) AS prev_day,
       amount - LAG(amount) OVER (PARTITION BY store ORDER BY day) AS change,
       LEAD(amount) OVER (PARTITION BY store ORDER BY day) AS next_day
FROM sales

The first row of each partition has no previous row, so LAG gives NULL there. LEAD gives NULL on the last row. These are used for “month-on-month change” and “days since the previous order”.

SUM(amount) OVER (PARTITION BY store ORDER BY day)   -- running total
SUM(amount) OVER (PARTITION BY store)                -- no ORDER BY → the store's full total on every row
The ORDER BY is what makes it “running”. Without it, you get the group total.



For a moving average, you choose how many rows go into the window (the “frame”):
AVG(amount) OVER (PARTITION BY store ORDER BY day
                  ROWS BETWEEN 1 PRECEDING AND CURRENT ROW)   -- 2-day moving average

SELECT class, SUM(mark) AS total,
       RANK() OVER (ORDER BY SUM(mark) DESC) AS rk,
       100.0 * SUM(mark) / SUM(SUM(mark)) OVER () AS pct_of_all
FROM exam
GROUP BY class

WINDOW FUNCTIONS: keep every row, add a calculated column
FUNC() OVER (PARTITION BY grp ORDER BY col)
  PARTITION BY = groups without squashing (leave out → whole table)
  ORDER BY     = order inside each group (needed for rank / LAG / running)

AGGREGATES     AVG(x) OVER (PARTITION BY g)      group avg on each row
               100.0*x / SUM(x) OVER ()          % of total
RANKING        ROW_NUMBER  1,2,3,4  exactly N rows, ties broken arbitrarily
               RANK        1,2,2,4  ties share, then skip
               DENSE_RANK  1,2,2,3  ties share, no gaps
LAG / LEAD     LAG(x) OVER (PARTITION BY g ORDER BY t)   previous row; first row → NULL
RUNNING        SUM(x) OVER (PARTITION BY g ORDER BY t)   no ORDER BY → group total instead
MOVING AVG     AVG(x) OVER (ORDER BY t ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
ON GROUPS      RANK() OVER (ORDER BY SUM(x) DESC) ... GROUP BY g
               SUM(SUM(x)) OVER ()   = total of the group totals

TOP N PER GROUP
WITH r AS (SELECT ..., ROW_NUMBER() OVER (PARTITION BY g ORDER BY x DESC) AS rn FROM t)
SELECT * FROM r WHERE rn <= N
- Windows run at SELECT → can't filter them in WHERE → wrap in a CTE
- Ties matter? Use RANK / DENSE_RANK, not ROW_NUMBER

In [5]:
# Dates in our tables are text like '2026-03-14'. SUBSTR(date, 1, 7) takes the first 7 characters, giving '2026-03'. Proper date functions come in 3.6.le

In [6]:
q("""SELECT name, class, mark, RANK() OVER (PARTITION BY class ORDER BY mark DESC) AS rk FROM exam""")
# q("SELECT * FROM exam ORDER BY class, mark DESC")
# (a) What are Bob’s and Cal’s ranks?
# 2 both
# (b) What is Dan’s rank, and why isn’t it 3?
# 4 -- 3 is technically take by cal
# (c) What rank does Eve get, given she has the top mark overall?
# 1

,name,class,mark,rk
0,Amy,A,90,1
1,Bob,A,85,2
2,Cal,A,85,2
3,Dan,A,70,4
4,Eve,B,95,1
5,Fay,B,80,2
6,Gus,B,80,2


In [7]:
q("""SELECT name, class, mark,
       AVG(mark) OVER (PARTITION BY class) AS cls_avg,
       mark - AVG(mark) OVER (PARTITION BY class) AS diff
FROM exam""")

# (a) How many rows come back?
# 7
# (b) What are the two class averages?
# 82.5, 85
# (c) Who is furthest below their class average?
# dan

,name,class,mark,cls_avg,diff
0,Amy,A,90,82.5,7.5
1,Bob,A,85,82.5,2.5
2,Cal,A,85,82.5,2.5
3,Dan,A,70,82.5,-12.5
4,Eve,B,95,85.0,10.0
5,Fay,B,80,85.0,-5.0
6,Gus,B,80,85.0,-5.0


In [8]:
q("""SELECT store, day, amount,
       SUM(amount) OVER (PARTITION BY store ORDER BY day) AS running,
       SUM(amount) OVER (PARTITION BY store) AS total
FROM sales""")

# (a) What is S1’s running total on day 3?
# 370
# (b) What does the total column show on day 1 for S2?
# 310

,store,day,amount,running,total
0,S1,1,100,100,570
1,S1,2,150,250,570
2,S1,3,120,370,570
3,S1,4,200,570,570
4,S2,1,80,80,310
5,S2,2,80,160,310
6,S2,3,90,250,310
7,S2,4,60,310,310


In [9]:
q("""SELECT store, day, amount,
       LAG(amount) OVER (PARTITION BY store ORDER BY day) AS prev,
       amount - LAG(amount) OVER (PARTITION BY store ORDER BY day) AS chg
FROM sales
""")

# (a) What do prev and chg show on day 1 of each store?
# nan
# (b) What is S2’s biggest drop, and on which day?
# day 4, -30

,store,day,amount,prev,chg
0,S1,1,100,NaN,NaN
1,S1,2,150,100.0,50.0
2,S1,3,120,150.0,-30.0
3,S1,4,200,120.0,80.0
4,S2,1,80,NaN,NaN
5,S2,2,80,80.0,0.0
6,S2,3,90,80.0,10.0
7,S2,4,60,90.0,-30.0


In [26]:
# q("""SELECT name, mark, ROW_NUMBER() OVER (ORDER BY mark DESC) AS rn 
# FROM exam 
# WHERE rn <= 3""")
# (a) Predict what happens.
#error on rn
# (b) Rewrite it with a CTE so it works.

q("""WITH cte AS(
SELECT name, mark, ROW_NUMBER() OVER (ORDER BY mark DESC) AS rn 
FROM exam )

SELECT * 
FROM cte
WHERE rn <= 3""")

,name,mark,rn
0,Eve,95,1
1,Amy,90,2
2,Bob,85,3


In [11]:
q("""WITH r AS (
    SELECT name, class, mark, RANK() OVER (PARTITION BY class ORDER BY mark DESC) AS rk
    FROM exam
)
SELECT * FROM r WHERE rk = 2""")


# (a) How many rows come back?
# 4
# (b) Would ROW_NUMBER give the same rows? Why not?
# nope, it would only return 2 rows with 2 rank for each class

,name,class,mark,rk
0,Bob,A,85,2
1,Cal,A,85,2
2,Fay,B,80,2
3,Gus,B,80,2


In [12]:
# B1. Beat your brand, the window way

# (a) Show every post with its brand’s average likes next to it, using AVG() OVER.
q("""WITH brand_avgs AS(
SELECT post_id,likes, AVG(likes) OVER(PARTITION BY brand) AS brand_avg
FROM posts)

SELECT COUNT(post_id)
FROM brand_avgs 
WHERE likes>brand_avg
""")
# (b) Wrap it in a CTE and count the posts above their brand average.
# (c) Does it match B5 from 3.4?
#yes

,COUNT(post_id)
0,104


In [13]:
# B2. Brand league table

# (a) For each brand, show total likes, % of all likes (1 dp) and rank, in one query using GROUP BY plus windows (note 8).
q("""
SELECT 
    brand, SUM(likes) AS total_likes, ROUND(100.0 * SUM(likes) / SUM(SUM(likes)) OVER (), 1) AS pct_of_all_likes,
    RANK() OVER (ORDER BY SUM(likes) DESC) AS brand_rank
FROM posts 
GROUP BY brand
""")
# (b) Which brand has about 40% of all likes?
# SnackBox (39.7%)

,brand,total_likes,pct_of_all_likes,brand_rank
0,SnackBox,161327,39.7,1
1,Glowly,101657,25.0,2
2,FitFuel,66044,16.2,3
3,UrbanThread,47147,11.6,4
4,HomeNest,30309,7.5,5


In [14]:
# B3. Top 3 posts per brand

# (a) Use ROW_NUMBER to rank posts within each brand by likes.
q("""WITH ranks AS(
SELECT post_id, brand, likes, ROW_NUMBER() OVER (PARTITION BY brand ORDER BY likes DESC) AS rnk
FROM posts)

SELECT * FROM ranks
WHERE rnk<=3""")
# (b) Keep the top 3 per brand.
# (c) How many rows should you get? Check.
#3*5= 15
# (d) Which brand’s #3 post beats every other brand’s #1?
#Snackbox

,post_id,brand,likes,rnk
0,P0190,FitFuel,4970,1
1,P0276,FitFuel,3435,2
2,P0003,FitFuel,3158,3
3,P0210,Glowly,5761,1
4,P0240,Glowly,5233,2
5,P0043,Glowly,5056,3
6,P0180,HomeNest,2032,1
7,P0258,HomeNest,1874,2
8,P0010,HomeNest,1801,3
9,P0019,SnackBox,13577,1


In [15]:
# B4. Second-best per platform

# (a) Find the 2nd most-liked post on each platform (post_id and likes).
q("""WITH ranks AS(
SELECT platform, post_id, likes, ROW_NUMBER() OVER (PARTITION BY platform ORDER BY likes DESC) AS rnk
FROM posts)

SELECT platform, post_id, rnk, likes FROM ranks
WHERE rnk<=2""")

# (b) Which platform’s 2nd-best post has more likes than another platform’s #1 post?
#TikTok

,platform,post_id,rnk,likes
0,Facebook,P0008,1,2304
1,Facebook,P0023,2,1638
2,Instagram,P0277,1,6133
3,Instagram,P0240,2,5233
4,TikTok,P0019,1,13577
5,TikTok,P0103,2,12052


In [16]:
# B5. First order per customer

# (a) Using ROW_NUMBER partitioned by customer and ordered by order_date, find each customer’s first order.
q("""WITH cust_ordered AS(
    SELECT customer_id, order_date, ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date) AS rnk
FROM orders)

SELECT * FROM cust_ordered
WHERE rnk= 1
""")

# (b) How many rows? Does it equal the number of distinct customers in orders?
# it is equal. 118 customers have orders, plus C121–C123 with no customer record, gives 121 distinct customer_ids in orders. So there’s no extra row. A related point to watch: C005, C039 and C116 each have two orders on the same date, so ROW_NUMBER picks one of the pair arbitrarily. If “first order” ties matter, you’d add a tiebreaker such as ORDER BY order_date, order_id.
# (c) What date was C041’s first order?
# 2026-01-02	

,customer_id,order_date,rnk
0,C001,2026-01-08,1
1,C002,2026-01-22,1
2,C003,2026-01-19,1
3,C004,2026-02-11,1
4,C005,2026-03-09,1
...,...,...,...
116,C119,2026-02-19,1
117,C120,2026-04-12,1
118,C121,2026-04-02,1
119,C122,2026-02-26,1


In [17]:
# B6. Month-on-month likes

# (a) In a CTE, total likes per month using SUBSTR(date, 1, 7).
q("""WITH month_likes AS (
SELECT 
    SUBSTR(date, 1, 7) AS month,
    SUM(likes) AS total_likes
FROM posts
GROUP BY SUBSTR(date, 1, 7))

SELECT month, LAG(total_likes) OVER (ORDER BY month) AS prev_month_likes, total_likes- LAG(total_likes) OVER (ORDER BY month) AS change
FROM month_likes

""")
# (b) Add the previous month’s likes and the change, using LAG.
# (c) Which month had the biggest increase? Which had the biggest drop?
#march, max increase, feb, max drop

,month,prev_month_likes,change
0,2026-01,NaN,NaN
1,2026-02,78103.0,-21301.0
2,2026-03,56802.0,23483.0
3,2026-04,80285.0,-21135.0
4,2026-05,59150.0,1809.0
5,2026-06,60959.0,10226.0


In [18]:
# B7. Running revenue

# (a) In a CTE, find delivered revenue per month (REPLACE/CAST trick, 2 dp).
q("""
WITH total_revenue AS(
    SELECT SUBSTR(order_date, 1, 7) AS month, SUM(CAST(REPLACE(unit_price, '$', '') AS FLOAT) * quantity) AS rev
FROM orders
    WHERE status='delivered'
GROUP BY SUBSTR(order_date, 1, 7)
)

SELECT month, rev, SUM(rev) OVER (ORDER BY month ASC) AS running
FROM total_revenue

""")
# (b) Add a running total.
# (c) Sanity check: what does the final running total equal? Have you seen that number before?
# 42,095.35 is the Phase 2 gate’s delivered total

,month,rev,running
0,2026-01,7515.66,7515.66
1,2026-02,5043.16,12558.82
2,2026-03,5812.59,18371.41
3,2026-04,7222.66,25594.07
4,2026-05,9257.11,34851.18
5,2026-06,7244.17,42095.35


In [19]:
# B8. Top customers per country, with ties

# (a) Count orders per customer per country (join to customers).
q("""
WITH customer_order_counts AS (
    SELECT 
        c.country, 
        o.customer_id, 
        COUNT(o.order_id) AS total_orders
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    GROUP BY c.country, o.customer_id
),
ranked_customers AS (
    SELECT 
        country, 
        customer_id, 
        total_orders,
        DENSE_RANK() OVER (PARTITION BY country ORDER BY total_orders DESC) AS rnk
    FROM customer_order_counts
)
SELECT 
    country, 
    customer_id, 
    total_orders
FROM ranked_customers
WHERE rnk = 1
""")

# (b) Use DENSE_RANK within each country to find the customer(s) with the most orders.
# (c) Which country has a three-way tie, and how many orders do they each have?
# FR - 5, UK - 8 
# (d) Why is DENSE_RANK (or RANK) better here than ROW_NUMBER?
# We need to know ties 

,country,customer_id,total_orders
0,DE,C041,10
1,FR,C038,5
2,FR,C085,5
3,FR,C114,5
4,UK,C052,8
5,UK,C079,8
6,UK,C094,8
7,US,C044,8


In [20]:
# B9. TikTok running count

# (a) Count TikTok posts per month.
# (b) Add a running total of posts.
q("""
WITH tiktok_monthly AS (
    SELECT 
        SUBSTR(date, 1, 7) AS month,
        COUNT(post_id) AS posts_count
    FROM posts
    WHERE platform = 'TikTok'   
    GROUP BY SUBSTR(date, 1, 7)
)
SELECT 
    month,
    posts_count,
    SUM(posts_count) OVER (ORDER BY month ASC) AS running_total_posts
FROM tiktok_monthly
ORDER BY month ASC
""")
# (c) Does the final total match TikTok’s count from 3.4 B3?
#yes

,month,posts_count,running_total_posts
0,2026-01,18,18
1,2026-02,14,32
2,2026-03,21,53
3,2026-04,13,66
4,2026-05,17,83
5,2026-06,21,104


In [21]:
q("""WITH cte AS (
    SELECT post_id, brand, likes,
       ROW_NUMBER() OVER (PARTITION BY brand ORDER BY likes DESC) AS rn
FROM posts
)
SELECT post_id,	brand,	likes FROM cte
WHERE rn = 1
""")
#cte needs to be defined for filtering


,post_id,brand,likes
0,P0190,FitFuel,4970
1,P0210,Glowly,5761
2,P0180,HomeNest,2032
3,P0019,SnackBox,13577
4,P0271,UrbanThread,3384


In [22]:
q("""WITH r AS (
    SELECT name, class, mark,
    DENSE_RANK() OVER (PARTITION BY class ORDER BY mark DESC) AS rn FROM exam
    )
SELECT * FROM r WHERE rn = 1
""")

#ROW_NUMBER gives each class exactly one #1, so a tied top scorer gets silently dropped.

,name,class,mark,rn
0,Amy,A,90,1
1,Eve,B,95,1


In [23]:
q("""WITH r AS (
SELECT name, class, mark,
DENSE_RANK() OVER (PARTITION BY class ORDER BY mark DESC) AS rn FROM exam)
           
SELECT * FROM r WHERE rn = 1""")
#parititon not defined

,name,class,mark,rn
0,Amy,A,90,1
1,Eve,B,95,1


In [24]:
q("""SELECT store, day, amount, SUM(amount) OVER (PARTITION BY store ORDER BY day) AS running
FROM sales""")
#day order not defined

,store,day,amount,running
0,S1,1,100,100
1,S1,2,150,250
2,S1,3,120,370
3,S1,4,200,570
4,S2,1,80,80
5,S2,2,80,160
6,S2,3,90,250
7,S2,4,60,310


In [25]:
# Check (no running, 4/5 to pass)

# Q1. Marks 90, 85, 85, 70, ordered highest first. What does RANK() give?
# a) 1, 2, 3, 4 b) 1, 2, 2, 3 c) 1, 2, 2, 4 d) 1, 1, 2, 3
# c

# Q2. Same marks. What does DENSE_RANK() give?
# a) 1, 2, 3, 4 b) 1, 2, 2, 3 c) 1, 2, 2, 4 d) 1, 1, 2, 3
# b

# Q3. (Write) Why can’t you write WHERE rn <= 3 in the same query that creates rn with ROW_NUMBER? What do you do instead?
# running order issue, we use a cte then filter

# Q4. LAG(amount) OVER (PARTITION BY store ORDER BY day). What is it on each store’s first day?
# a) 0 b) the same day’s amount c) NULL d) the previous store’s last amount
# c

# Q5. (Write) Write a query that returns the top 2 posts by likes on each platform (post_id, platform, likes).
q("""WITH rnks AS(SELECT post_id, likes, platform, ROW_NUMBER() OVER (PARTITION BY platform ORDER BY likes DESC) AS rnk
FROM posts)

SELECT * FROM rnks WHERE rnk<=2""")





,post_id,likes,platform,rnk
0,P0008,2304,Facebook,1
1,P0023,1638,Facebook,2
2,P0277,6133,Instagram,1
3,P0240,5233,Instagram,2
4,P0019,13577,TikTok,1
5,P0103,12052,TikTok,2
